In [19]:
import sqlite3
from pathlib import Path

import numpy as np
import pandas as pd
import folium
from folium.plugins import TimestampedGeoJson

database_path = Path("../data/db/edition.sqlite")

def read_df(sql: str, params=()):
    with sqlite3.connect(database_path) as conn:
        return pd.read_sql_query(sql, conn, params=params)


In [20]:
period_map_path = Path("period_mapping.csv")  # ggf. anpassen
df_map = pd.read_csv(period_map_path)

# Minimal absichern:
needed = {"period_raw","period_key","period_norm","date_from","date_to"}
missing = needed - set(df_map.columns)
if missing:
    raise ValueError(f"period_mapping.csv fehlt Spalten: {missing}")

df_map.head()


,period_raw,period_key,period_norm,date_from,date_to,certainty,is_transition,has_question_mark,note
0,L.B.A,L.B.A,Late Bronze Age,-1600,-1300,high,False,False,Approx.; LBA variably defined in SE Arabia.
1,U.Nar,U.Nar,Umm an-Nar,-2600,-2000,high,False,False,Classic Umm an-Nar horizon.
2,W.Suq,W.Suq,Wadi Suq,-2000,-1600,high,False,False,Approx.; overlaps with late Umm an-Nar / early...
3,IA,IA,Iron Age (unspecified),-1300,-300,medium,False,False,Broad umbrella; prefer EIA/MIA/LIA where possi...
4,EIA,EIA,Early Iron Age,-1300,-900,high,False,False,Approx.; subdivisions vary by author.


In [21]:
sql_site_period = """
SELECT
  s.id AS site_id,
  s.code AS site_code,
  s.name AS site_name,
  v.name AS village_name,
  c.latitude,
  c.longitude,
  pc.period AS period_raw,
  SUM(COALESCE(pcs.sherd_count, 0)) AS sherds
FROM pottery_class_site pcs
JOIN pottery_class pc ON pc.id = pcs.pottery_class_id
JOIN site s ON s.id = pcs.site_id
LEFT JOIN village v ON v.id = s.village_id
JOIN coords c ON c.site_id = s.id
WHERE c.latitude IS NOT NULL
  AND c.longitude IS NOT NULL
  AND c.latitude BETWEEN -90 AND 90
  AND c.longitude BETWEEN -180 AND 180
GROUP BY
  s.id, s.code, s.name, v.name, c.latitude, c.longitude, pc.period
ORDER BY s.code, pc.period;
"""

df_sp = read_df(sql_site_period)
df_sp.head(), df_sp.shape


(   site_id site_code         site_name       village_name   latitude  \
 0        1    CS.1.1  AI-Fulayj CS.1.1  AL-FULAYJ VILLAGE  22.888825   
 1        1    CS.1.1  AI-Fulayj CS.1.1  AL-FULAYJ VILLAGE  22.888825   
 2        4    CS.1.4  AI-Fulayj CS.1.4  AL-FULAYJ VILLAGE  22.866359   
 3        4    CS.1.4  AI-Fulayj CS.1.4  AL-FULAYJ VILLAGE  22.866359   
 4        4    CS.1.4  AI-Fulayj CS.1.4  AL-FULAYJ VILLAGE  22.866359   
 
    longitude            period_raw  sherds  
 0  58.061360             L.Islamic       3  
 1  58.061360                 U.Nar       2  
 2  58.050853              Islamic?       1  
 3  58.050853             L.Islamic       3  
 4  58.050853  L.Islamic to Recent?       4  ,
 (152, 8))

In [22]:
# Join: period_raw (DB) -> period_mapping period_raw
df = df_sp.merge(df_map, on="period_raw", how="left")

# Diagnose: unmapped Perioden
unmapped = df[df["period_norm"].isna()]["period_raw"].dropna().unique()
if len(unmapped):
    print("WARN: Unmapped period_raw values:", unmapped)

# Chronologie: sort_key als date_from, sekundär date_to
# (date_from/date_to sind int; negative Werte ok)
df["date_from"] = pd.to_numeric(df["date_from"], errors="coerce")
df["date_to"]   = pd.to_numeric(df["date_to"], errors="coerce")

# Für Slider-Reihenfolge: eindeutige Perioden nach date_from sortieren
period_order = (
    df_map.assign(date_from=pd.to_numeric(df_map["date_from"], errors="coerce"),
                  date_to=pd.to_numeric(df_map["date_to"], errors="coerce"))
         .sort_values(["date_from","date_to","period_key"])
         ["period_raw"]
         .tolist()
)

period_order[:10], len(period_order)


(['U.Nar',
  'W.Suq',
  'L.B.A',
  'EIA',
  'IA',
  'LIA',
  'E.Islamic',
  'Islamic?',
  'M.Islamic',
  'M.Islamic to L.Islamic'],
 13)

In [23]:
start = pd.Timestamp("2000-01-01")
period_to_time = {p: (start + pd.DateOffset(months=i)).strftime("%Y-%m-%d") for i, p in enumerate(period_order)}

# Optional: Kontrolle
list(period_to_time.items())[:5]


[('U.Nar', '2000-01-01'),
 ('W.Suq', '2000-02-01'),
 ('L.B.A', '2000-03-01'),
 ('EIA', '2000-04-01'),
 ('IA', '2000-05-01')]

In [30]:
df_map_idx = df_map.set_index("period_raw")

time_to_label = {}
for p in period_order:
    t = period_to_time[p]
    row = df_map_idx.loc[p]
    label = f"{row['period_norm']} ({int(row['date_from'])}…{int(row['date_to'])})"
    time_to_label[t] = label

list(time_to_label.items())[:3]


[('2000-01-01', 'Umm an-Nar (-2600…-2000)'),
 ('2000-02-01', 'Wadi Suq (-2000…-1600)'),
 ('2000-03-01', 'Late Bronze Age (-1600…-1300)')]

In [31]:
import json
from branca.element import MacroElement, Template

def add_period_labels_to_slider(folium_map, time_to_label: dict, position="bottomleft"):
    """
    Robuste Variante:
    - blendet die Standard-Datumsanzeige der TimeDimension-Control aus
    - zeigt stattdessen ein eigenes Periodenlabel (Leaflet-Control) an
    """
    mapping_json = json.dumps(time_to_label)

    tmpl = Template(f"""
    {{% macro script(this, kwargs) %}}
    (function() {{
        var map = {folium_map.get_name()};
        var mapping = {mapping_json};

        // 1) CSS: Original-Datumsanzeige ausblenden (klassenabhängig, daher breit gefasst)
        var style = document.createElement('style');
        style.innerHTML = `
            .timecontrol-date {{ display: none !important; }}
            .leaflet-control-timecontrol .timecontrol-date {{ display: none !important; }}
        `;
        document.head.appendChild(style);

        // 2) Eigenes Label als Leaflet-Control
        function makeLabelControl() {{
            var LabelControl = L.Control.extend({{
                options: {{ position: "{position}" }},
                onAdd: function() {{
                    var div = L.DomUtil.create('div', 'period-label-control');
                    div.id = 'period-label-control';
                    div.style.background = 'white';
                    div.style.padding = '8px 10px';
                    div.style.border = '1px solid #ccc';
                    div.style.borderRadius = '6px';
                    div.style.boxShadow = '0 1px 3px rgba(0,0,0,0.15)';
                    div.style.fontSize = '12px';
                    div.style.maxWidth = '320px';
                    div.innerHTML = '<b>Periode</b><br><span id="period-label-text">…</span>';

                    // verhindert, dass Drag/Scroll auf dem Label die Karte beeinflusst
                    L.DomEvent.disableClickPropagation(div);
                    L.DomEvent.disableScrollPropagation(div);
                    return div;
                }}
            }});
            return new LabelControl();
        }}

        function toKey(t) {{
            // TimeDimension liefert Zeit meist in ms seit Epoch (number) oder Date (je nach Version)
            if (typeof t === "number") {{
                var d = new Date(t);
                var yyyy = d.getFullYear().toString().padStart(4,'0');
                var mm = (d.getMonth()+1).toString().padStart(2,'0');
                var dd = d.getDate().toString().padStart(2,'0');
                return yyyy + "-" + mm + "-" + dd;
            }}
            if (t instanceof Date) {{
                var yyyy = t.getFullYear().toString().padStart(4,'0');
                var mm = (t.getMonth()+1).toString().padStart(2,'0');
                var dd = t.getDate().toString().padStart(2,'0');
                return yyyy + "-" + mm + "-" + dd;
            }}
            if (typeof t === "string") {{
                return t.slice(0, 10);
            }}
            return null;
        }}

        function setLabel() {{
            if (!map.timeDimension) return;
            var t = map.timeDimension.getCurrentTime();
            var key = toKey(t);
            if (!key) return;

            var label = mapping[key] || key;
            var el = document.getElementById('period-label-text');
            if (el) el.innerHTML = label;
        }}

        function hook() {{
            if (!map.timeDimension) {{
                setTimeout(hook, 200);
                return;
            }}

            // Control einmalig hinzufügen
            if (!document.getElementById('period-label-control')) {{
                map.addControl(makeLabelControl());
            }}

            // initial setzen
            setLabel();

            // zuverlässig updaten
            map.timeDimension.on('timechange', function() {{ setLabel(); }});
            map.timeDimension.on('timeload', function() {{ setLabel(); }});
        }}

        hook();
    }})();
    {{% endmacro %}}
    """)

    macro = MacroElement()
    macro._template = tmpl
    folium_map.get_root().add_child(macro)
    return folium_map


In [32]:
def radius_scale(values, r_min=3, r_max=14):
    v = np.array(values, dtype=float)
    v = np.where(v < 0, 0, v)
    v = np.log1p(v)
    if v.max() == v.min():
        return lambda x: (r_min + r_max) / 2
    return lambda x: r_min + (np.log1p(max(x,0)) - v.min()) * (r_max - r_min) / (v.max() - v.min())

r = radius_scale(df["sherds"].fillna(0), r_min=3, r_max=12)

# Einfache Farbpalette pro Period_key
def palette(n):
    base = [
        "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd",
        "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#17becf",
        "#393b79", "#637939", "#8c6d31", "#843c39", "#7b4173"
    ]
    return [base[i % len(base)] for i in range(n)]

# Farben pro period_raw entlang period_order
colors = {p: palette(len(period_order))[i] for i, p in enumerate(period_order)}

features = []
for _, row in df.iterrows():
    p_raw = row["period_raw"]
    if pd.isna(p_raw) or p_raw not in period_to_time:
        continue

    t = period_to_time[p_raw]
    col = colors.get(p_raw, "#000000")

    # echte Jahresinfos aus CSV:
    date_from = row.get("date_from")
    date_to = row.get("date_to")
    period_norm = row.get("period_norm") if pd.notna(row.get("period_norm")) else p_raw

    popup = (
        f"<b>{row['site_code']}</b><br>"
        f"{row['site_name'] or ''}<br>"
        f"Village: {row['village_name'] or ''}<br>"
        f"Period: <b>{period_norm}</b> ({p_raw})<br>"
        f"Range: {int(date_from) if pd.notna(date_from) else '?'} … {int(date_to) if pd.notna(date_to) else '?'}<br>"
        f"Sherds: {int(row['sherds'])}"
    )

    features.append({
        "type": "Feature",
        "geometry": {"type": "Point", "coordinates": [row["longitude"], row["latitude"]]},
        "properties": {
            "time": t,
            "popup": popup,
            "style": {"color": col},
            "icon": "circle",
            "iconstyle": {
                "fillColor": col,
                "fillOpacity": 0.7,
                "stroke": "true",
                "radius": float(r(row["sherds"]))
            }
        }
    })

geojson = {"type": "FeatureCollection", "features": features}
len(features)


152

In [33]:
center = [df["latitude"].mean(), df["longitude"].mean()]

m_anim = folium.Map(location=center, zoom_start=8, tiles="OpenStreetMap")

TimestampedGeoJson(
    geojson,
    period="P1M",              
    add_last_point=False,
    auto_play=False,
    loop=False,
    max_speed=2,
    loop_button=True,
    date_options="YYYY-MM",     # synthetische Anzeige
    time_slider_drag_update=True,
).add_to(m_anim)

m_anim = add_period_labels_to_slider(m_anim, time_to_label)
m_anim
